In [9]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/world_bank_cleaned.csv")

df.head()

,Country Name,Country Code,adolescent_fertility,gdp_per_capita,female_labor_force_participation,female_life_expectancy,maternal_mortality,women_in_parliament,female_secondary_enrollment,female_tertiary_enrollment,female_unemployment
0,Afghanistan,AFG,64.068,413.757895,5.145,67.536,521.0,27.016129,45.159691,5.936540,26.597
1,Albania,ALB,12.789,9730.869219,57.963,81.446,7.0,35.714286,92.668293,77.031527,10.957
2,Algeria,DZA,8.698,5370.477235,14.338,77.696,62.0,7.862408,103.220718,67.258843,21.064
3,American Samoa,ASM,34.181,18017.458938,NaN,75.838,NaN,NaN,NaN,NaN,NaN
4,Andorra,AND,3.480,46812.426101,NaN,86.107,11.0,50.000000,99.468742,68.003820,NaN


In [10]:
id_cols = ["Country Name", "Country Code"]

positive_indicators = [
    "female_secondary_enrollment",
    "female_tertiary_enrollment",
    "female_labor_force_participation",
    "female_life_expectancy",
    "women_in_parliament"
]

negative_indicators = [
    "female_unemployment",
    "maternal_mortality",
    "adolescent_fertility"
]

comparison_indicators = [
    "gdp_per_capita"
]

all_index_indicators = positive_indicators + negative_indicators

In [11]:
# removing  countries with too much missing data

df_index = df.copy()

df_index["missing_count"] = df_index[all_index_indicators].isna().sum(axis=1)
df_index["missing_percentage"] = df_index[all_index_indicators].isna().mean(axis=1) * 100

# Keeping countries with at least 60% of the index indicators available
df_index = df_index[df_index["missing_percentage"] <= 40].copy()

df_index[["Country Name", "missing_count", "missing_percentage"]].sort_values(
    "missing_percentage", ascending=False
).head()

,Country Name,missing_count,missing_percentage
6,Antigua and Barbuda,3,37.5
54,Dominica,3,37.5
113,Liechtenstein,3,37.5
101,Kiribati,3,37.5
179,St. Kitts and Nevis,3,37.5


In [12]:
#filling missing values with median 

for col in all_index_indicators:
    df_index[col] = df_index[col].fillna(df_index[col].median())

# GDP is comparison only, so we can also fill it for plotting later if needed
df_index["gdp_per_capita"] = df_index["gdp_per_capita"].fillna(df_index["gdp_per_capita"].median())

df_index = df_index.drop(columns=["missing_count", "missing_percentage"])

# Normalizing indicators 

def min_max_normalise(series):
    return (series - series.min()) / (series.max() - series.min())

for col in positive_indicators:
    df_index[col + "_norm"] = min_max_normalise(df_index[col])

for col in negative_indicators:
    df_index[col + "_norm"] = 1 - min_max_normalise(df_index[col])

In [13]:
#creating sub indices

df_index["education_index"] = df_index[
    [
        "female_secondary_enrollment_norm",
        "female_tertiary_enrollment_norm"
    ]
].mean(axis=1)

df_index["economic_participation_index"] = df_index[
    [
        "female_labor_force_participation_norm",
        "female_unemployment_norm"
    ]
].mean(axis=1)

df_index["health_index"] = df_index[
    [
        "female_life_expectancy_norm",
        "maternal_mortality_norm",
        "adolescent_fertility_norm"
    ]
].mean(axis=1)

df_index["political_representation_index"] = df_index[
    [
        "women_in_parliament_norm"
    ]
].mean(axis=1)



In [14]:
#finalized women's development index

df_index["womens_development_index"] = df_index[
    [
        "education_index",
        "economic_participation_index",
        "health_index",
        "political_representation_index"
    ]
].mean(axis=1)

df_index = df_index.sort_values("womens_development_index", ascending=False)

df_index[
    [
        "Country Name",
        "Country Code",
        "womens_development_index",
        "education_index",
        "economic_participation_index",
        "health_index",
        "political_representation_index",
        "gdp_per_capita"
    ]
].head(20)

,Country Name,Country Code,womens_development_index,education_index,economic_participation_index,health_index,political_representation_index,gdp_per_capita
66,Finland,FIN,0.829645,0.853936,0.759592,0.954033,0.751020,52834.291675
87,Iceland,ISL,0.815931,0.647533,0.883623,0.955116,0.777454,82138.789297
185,Sweden,SWE,0.815507,0.770771,0.768192,0.965213,0.757850,54950.283473
146,Norway,NOR,0.814351,0.712781,0.828024,0.963066,0.753532,87497.217965
10,Australia,AUS,0.800856,0.792299,0.827854,0.956158,0.627112,65058.377315
138,Netherlands,NLD,0.791219,0.733825,0.828912,0.949076,0.653061,63515.603078
140,New Zealand,NZL,0.790299,0.658514,0.850621,0.929412,0.722650,49075.956283
18,Belgium,BEL,0.789190,0.776405,0.727633,0.956123,0.696599,55291.475454
52,Denmark,DNK,0.785065,0.686410,0.788774,0.953639,0.711435,68043.546697
201,United Arab Emirates,ARE,0.764205,0.533030,0.752728,0.954736,0.816327,49850.687218


In [15]:

#results

df_index.to_csv("../data/processed/final_womens_development_index.csv", index=False)